# Exercise 4

## Imports

In [28]:
import math
from enum import Enum
from pathlib import Path
from typing import Dict, Optional

import librosa
import numpy as np
import pandas as pd
from pandas import DataFrame

## Constants

In [29]:
current_dir = Path.cwd()

# Music file
rock_audiofile_path = current_dir.joinpath("data", "rock.mp3")


class FeatureType(Enum):
    SPECTRAL_CENTROID = "spectral_centroid"
    ZERO_CROSSING_RATE = "zero_crossing_rate"
    ROOT_MEAN_SQUARED_ENERGY = "root_mean_squared_energy"

## Exercise 1

Load the file rock.mp3 from the week 2 exercises in Python and extract the following features with
librosa, once with a frame length of 512 and once with 760:
- Spectral centroid
- Zero-crossing rate
- Root mean squared energy (RMS)

See https://librosa.org/doc/latest/feature.html for documentation on how to extract the features.

In [44]:
def get_column_name(feature_type: FeatureType, frame_length: int) -> str:
    return f"{feature_type.value}-{frame_length}"


def extract_feature(feature_type: FeatureType, y: np.ndarray, sr: int, frame_length: int, hop_length: int) -> pd.Series:
    if feature_type is FeatureType.SPECTRAL_CENTROID:
        features = librosa.feature.spectral_centroid(y=y,
                                                     sr=sr,
                                                     n_fft=frame_length,
                                                     hop_length=hop_length, )
    elif feature_type is FeatureType.ZERO_CROSSING_RATE:
        features = librosa.feature.zero_crossing_rate(y=y,
                                                      hop_length=frame_length,
                                                      frame_length=hop_length, )
    elif feature_type is FeatureType.ROOT_MEAN_SQUARED_ENERGY:
        features = librosa.feature.rms(y=y,
                                       frame_length=frame_length,
                                       hop_length=hop_length, )
    else:
        raise ValueError(f"Unknown feature type: {feature_type}")

    return pd.Series(features.flatten(), name=get_column_name(feature_type, frame_length))


# Load audio file
y, sr = librosa.load(rock_audiofile_path)
# y -> audio time series
# sr -> sampling rate
n_samples = len(y)
print(f"n samples = {n_samples}")

# Feature dataframe
features_df = DataFrame()
feature_window_length_dict = {}

frame_lengths = [512, 760]
for frame_length in frame_lengths:
    print(f"Extracting features with frame length: {frame_length}")

    hop_length = frame_length  # Assuming no overlap

    for feature_type in FeatureType:
        new_feature_series = extract_feature(feature_type, y, sr, frame_length, hop_length)
        print(new_feature_series.shape)

        features_df = pd.concat([features_df, new_feature_series], axis=1)
        feature_window_length_dict[new_feature_series.name] = frame_length

print(features_df)
print(feature_window_length_dict)

n samples = 5003136
Extracting features with frame length: 512
(9772,)
(9772,)
(9772,)
Extracting features with frame length: 760
(6584,)
(6584,)
(6584,)
      spectral_centroid-512  zero_crossing_rate-512  \
0                  0.000000                0.000000   
1                  0.000000                0.000000   
2               6568.019614                0.000000   
3               5023.175337                0.000000   
4               4711.511930                0.210938   
...                     ...                     ...   
9767               0.000000                0.000000   
9768               0.000000                0.000000   
9769               0.000000                0.000000   
9770               0.000000                0.000000   
9771               0.000000                0.000000   

      root_mean_squared_energy-512  spectral_centroid-760  \
0                     0.000000e+00               0.000000   
1                     0.000000e+00            7406.149386   
2 

In [ ]:
# TODO: visualize data
#  create view of first few frames comparing feature values extracted with different window sizes (like in slide 18)

## Exercise 2

Given the features from Exercise 1, calculate a harmonised feature matrix for all six feature vectors.
The features should be harmonised to the frame length of fmin = 512.
First, use the harmonisation method shown on slide 18 and 19 of the lecture “Feature Processing and
Selection”.
Then, do the same by using a weighted average method in which features of adjacent frames are
weighted by their overlap ratio with the new frame.

In [50]:
def get_min_feature_length(feature_window_lengths: Dict[str, int]) -> int:
    return min(feature_window_lengths.values())


class HarmonizationMethod(Enum):
    SELECTION = "selection"
    AVERAGE = "average"


def harmonization_matrix(df: DataFrame, feature_window_lengths: Dict[str, int], total_samples: int,
                         harmonization_method: HarmonizationMethod = HarmonizationMethod.SELECTION,
                         min_feature_length: Optional[int] = None) -> DataFrame:
    """

    :param df: multiple series corresponding to different features.
        Can have trailing Nands.
        It is also assumed that each feature processes all samples.
    :param feature_window_lengths:
    :param min_feature_length:
    :return:
    """
    if min_feature_length is None:
        min_feature_length = get_min_feature_length(feature_window_lengths)

    num_windows = math.ceil(total_samples / min_feature_length)

    hm_df = pd.DataFrame()
    for series_name in df:
        window_length = feature_window_lengths[series_name]
        if window_length > min_feature_length:
            hf = []
            for w in range(num_windows):
                prev_opt_idx = (min_feature_length * (w - 1) // window_length) + 1
                next_opt_idx = prev_opt_idx + 1

                a = min_feature_length * (w - 1)
                b = window_length * prev_opt_idx
                c = min_feature_length * w

                w_first_len = b - a
                w_second_len = c - b

                if harmonization_method is HarmonizationMethod.SELECTION:
                    hf.append(df[series_name][prev_opt_idx]) if w_first_len > w_second_len else hf.append(
                        df[series_name][next_opt_idx])
                elif harmonization_method is HarmonizationMethod.AVERAGE:
                    hf.append((df[series_name][prev_opt_idx] * w_first_len + df[series_name][
                        next_opt_idx] * w_second_len) / min_feature_length)
            hm_df[series_name] = pd.Series(hf)
        else:
            hm_df[series_name] = df[series_name].copy()

    return hm_df, min_feature_length

## 2.1 - Harmonisation of feature matrix 1: SELECTION

In [54]:
hm1_df, min_feature_length = harmonization_matrix(features_df, feature_window_length_dict, n_samples,
                                                  HarmonizationMethod.SELECTION)
print(f"min_feature_length: {min_feature_length}")
print(hm1_df)

min_feature_length: 512
      spectral_centroid-512  zero_crossing_rate-512  \
0                  0.000000                0.000000   
1                  0.000000                0.000000   
2               6568.019614                0.000000   
3               5023.175337                0.000000   
4               4711.511930                0.210938   
...                     ...                     ...   
9767               0.000000                0.000000   
9768               0.000000                0.000000   
9769               0.000000                0.000000   
9770               0.000000                0.000000   
9771               0.000000                0.000000   

      root_mean_squared_energy-512  spectral_centroid-760  \
0                     0.000000e+00               0.000000   
1                     0.000000e+00            7406.149386   
2                     1.674096e-22            5039.873056   
3                     4.431757e-13            5039.873056   
4         

In [ ]:
# Todo, visualize results !

## 2.2 - Harmonisation of feature matrix 2: AVERAGE

In [55]:
hm2_df, min_feature_length = harmonization_matrix(features_df, feature_window_length_dict, n_samples,
                                                  HarmonizationMethod.AVERAGE)
print(f"min_feature_length: {min_feature_length}")
print(hm2_df)

min_feature_length: 512
      spectral_centroid-512  zero_crossing_rate-512  \
0                  0.000000                0.000000   
1                  0.000000                0.000000   
2               6568.019614                0.000000   
3               5023.175337                0.000000   
4               4711.511930                0.210938   
...                     ...                     ...   
9767               0.000000                0.000000   
9768               0.000000                0.000000   
9769               0.000000                0.000000   
9770               0.000000                0.000000   
9771               0.000000                0.000000   

      root_mean_squared_energy-512  spectral_centroid-760  \
0                     0.000000e+00               0.000000   
1                     0.000000e+00            8552.314483   
2                     1.674096e-22            6186.038153   
3                     4.431757e-13            5007.293462   
4         

In [ ]:
# Todo, visualize results !

In [ ]:
# Todo, visualize comparison of original, result1, result 2 (per feature?)

## Exercise 3

For each of three feature types, calculate the correlation between the features with frame length 512
from Exercise 1 and the harmonised features with original frame length 760.
Which harmonisation method achieves higher correlation with the extracted features?

In [ ]:
#get features with length 512
features_512 = [x for x in features_df if "512" in x]
#create correlation matrix for them
correlation_512 = features_df[features_512].corr()
print(correlation_512)

#TODO add correlation matrix here for the harmonisation

                              spectral_centroid-512  zero_crossing_rate-512  \
spectral_centroid-512                      1.000000                0.779382   
zero_crossing_rate-512                     0.779382                1.000000   
root_mean_squared_energy-512              -0.218181               -0.426609   

                              root_mean_squared_energy-512  
spectral_centroid-512                            -0.218181  
zero_crossing_rate-512                           -0.426609  
root_mean_squared_energy-512                      1.000000  
